# ShieldVoice (SIH26104) — Kaggle GPU Training Pipeline
### AI-Powered Real-Time Voice Anti-Spoofing & Deepfake Detection

**Hardware**: GPU Enabled (Tesla T4 x 2 / P100)
**Datasets Attached**:
- `awsaf49/asvpoof-2019-dataset`
- `birdy654/deep-voice-deepfake-voice-recognition`
- `mohammedabdeldayem/avsspoof-2021`
- `trapka/mlaadthe-multi-languagaudioanti-spoofing-dataset`

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count:  {torch.cuda.device_count()}")

## 2. Clone Codebase & Install Dependencies

In [ ]:
!rm -rf /kaggle/working/SIH
!git clone https://github.com/kishore-in2007/SIH.git /kaggle/working/SIH
%cd /kaggle/working/SIH

!pip install -r requirements.txt -q

## 3. Locate Native Kaggle Input Datasets

In [ ]:
import os
import glob

print("Scanning Kaggle Input Datasets in /kaggle/input/ ...")
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root[len("/kaggle/input"):].count(os.sep)
    if depth < 2:
        print(f"{root} ({len(files)} files, {len(dirs)} subdirs)")

## 4. Run GPU Training Pipeline

In [ ]:
# Train Wav2Vec2-XLS-R + AASIST Graph Attention Model
!python train.py \
    --kagglehub_download \
    --model_type wav2vec2_aasist \
    --ssl_model facebook/wav2vec2-xls-r-300m \
    --epochs 25 \
    --batch_size 16 \
    --lr 1e-4 \
    --ssl_lr 1e-5 \
    --device cuda \
    --save_dir /kaggle/working/saved_models

## 5. Benchmark Evaluation (Equal Error Rate & Accuracy)

In [ ]:
!python evaluate.py \
    --checkpoint /kaggle/working/saved_models/best_model.pt \
    --device cuda

## 6. Test Deepfake Voice Inference

In [ ]:
audio_files = glob.glob("/kaggle/input/**/*.flac", recursive=True) + glob.glob("/kaggle/input/**/*.wav", recursive=True)
if audio_files:
    test_sample = audio_files[0]
    print(f"Testing inference on: {test_sample}")
    !python inference.py --audio "{test_sample}" --checkpoint /kaggle/working/saved_models/best_model.pt
else:
    print("No sample audio found.")